# Build External Data Pipeline\n\nUse this notebook to test the first five external-data steps interactively before turning them into production ingestion jobs. The notebook is free-first and designed around SEC, FRED, GDELT, and optional FMP enrichment.

## 0. Setup\n\n- `SEC` is used for company master data, filing metadata, company facts, and filing text.\n- `FRED` is used for macro series.\n- `GDELT` is used for news and narrative discovery.\n- `FMP` is optional and only runs if `FMP_API_KEY` is set in the environment.

In [ ]:
from __future__ import annotations\n\nimport json\nimport os\nimport re\nfrom pathlib import Path\nfrom urllib.parse import quote\nfrom urllib.request import Request, urlopen\n\nimport pandas as pd\n\nROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()\nDATA_DIR = ROOT / 'data'\nEXTERNAL_DIR = DATA_DIR / 'external'\nINTERIM_DIR = DATA_DIR / 'interim'\nRAW_DIR = DATA_DIR / 'raw'\nfor path in [EXTERNAL_DIR / 'sec', EXTERNAL_DIR / 'fred', EXTERNAL_DIR / 'gdelt', EXTERNAL_DIR / 'fmp', INTERIM_DIR / 'vector_preview']:\n    path.mkdir(parents=True, exist_ok=True)\n\nSEC_HEADERS = {\n    'User-Agent': 'portfolio-analyzer research contact@example.com',\n    'Accept-Encoding': 'gzip, deflate',\n}\n\ndef fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:\n    request = Request(url, headers=headers or {})\n    with urlopen(request) as response:\n        return json.loads(response.read().decode('utf-8'))\n\ndef fetch_text(url: str, headers: dict[str, str] | None = None) -> str:\n    request = Request(url, headers=headers or {})\n    with urlopen(request) as response:\n        return response.read().decode('utf-8', errors='ignore')\n\ndef simple_chunk(text: str, chunk_size: int = 1400) -> list[str]:\n    clean = re.sub(r'\s+', ' ', text).strip()\n    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]\n\ndef html_to_text(html: str) -> str:\n    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)\n    no_tags = re.sub(r'<[^>]+>', ' ', no_script)\n    return re.sub(r'\s+', ' ', no_tags).strip()\n\nfake_csv = RAW_DIR / 'fake_mantis_invest.csv'\ntransactions = pd.read_csv(fake_csv) if fake_csv.exists() else pd.DataFrame()\ntickers = sorted({str(t).strip() for t in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist() if str(t).strip()})\ntickers[:10], len(tickers)

## 1. Master Data\n\nBuild the ticker universe, map it to SEC CIKs, and create the company master table. This becomes the base join key for the rest of the pipeline.

In [ ]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'\ncompany_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)\ncompany_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})\ncompany_master['ticker'] = company_master['ticker'].astype(str).str.upper()\ncompany_master['cik'] = company_master['cik'].astype(str).str.zfill(10)\nif tickers:\n    company_master = company_master[company_master['ticker'].isin(tickers)].copy()\ncompany_master.to_parquet(EXTERNAL_DIR / 'sec' / 'company_master.parquet', index=False)\ncompany_master.head(10)

## 2. Structured Core\n\nStart with macro series from FRED and optionally enrich ticker metadata with FMP if an API key is available.

In [ ]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')\nfred_series = {\n    'FEDFUNDS': 'Fed Funds Rate',\n    'CPIAUCSL': 'CPI',\n    'UNRATE': 'Unemployment Rate',\n    'DGS10': '10Y Treasury',\n    'DGS2': '2Y Treasury',\n}\n\nfred_frames = []\nfor series_id, title in fred_series.items():\n    fred_url = (\n        'https://api.stlouisfed.org/fred/series/observations'\n        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'\n    )\n    payload = fetch_json(fred_url)\n    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]\n    frame['series_id'] = series_id\n    frame['title'] = title\n    fred_frames.append(frame)\n\nfred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()\nfred_df.to_parquet(EXTERNAL_DIR / 'fred' / 'macro_series.parquet', index=False)\nfred_df.tail(10)

In [ ]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')\nif FMP_API_KEY and not company_master.empty:\n    sample_tickers = ','.join(company_master['ticker'].head(5).tolist())\n    fmp_url = f'https://financialmodelingprep.com/api/v3/profile/{sample_tickers}?apikey={FMP_API_KEY}'\n    fmp_profiles = pd.DataFrame(fetch_json(fmp_url))\n    fmp_profiles.to_parquet(EXTERNAL_DIR / 'fmp' / 'company_profiles_preview.parquet', index=False)\n    display(fmp_profiles[['symbol', 'companyName', 'sector', 'industry', 'mktCap']].head())\nelse:\n    print('FMP_API_KEY not set. Skip this cell for now or add a free-tier key to test profile enrichment.')

## 3. SEC Filing Metadata + Company Facts\n\nPull official filing metadata and company facts for one sample ticker so you can validate the SEC ingestion path before scaling it.

In [ ]:
sample_ticker = company_master['ticker'].iloc[0] if not company_master.empty else 'AAPL'\nsample_cik = company_master.loc[company_master['ticker'].eq(sample_ticker), 'cik'].iloc[0] if not company_master.empty else '0000320193'\n\nsubmissions_url = f'https://data.sec.gov/submissions/CIK{sample_cik}.json'\nsubmissions = fetch_json(submissions_url, headers=SEC_HEADERS)\nrecent = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))\nrecent = recent[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].head(20)\nrecent.to_parquet(EXTERNAL_DIR / 'sec' / f'{sample_ticker.lower()}_filings_preview.parquet', index=False)\ndisplay(recent.head(10))\n\nfacts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{sample_cik}.json'\ncompany_facts = fetch_json(facts_url, headers=SEC_HEADERS)\nus_gaap = company_facts.get('facts', {}).get('us-gaap', {})\nfact_names = list(us_gaap.keys())[:15]\npd.DataFrame({'ticker': [sample_ticker] * len(fact_names), 'fact_tag': fact_names})

## 4. Text Pipeline / Vector Preview\n\nTake one recent 10-K or 10-Q, fetch the filing text, clean it, and create vector-ready chunks with metadata. This is a preview of what will later go into the vector database.

In [ ]:
candidate_forms = recent[recent['form'].isin(['10-K', '10-Q'])].copy()\nif candidate_forms.empty:\n    raise ValueError(f'No recent 10-K/10-Q found for {sample_ticker}')\n\nfiling_row = candidate_forms.iloc[0]\naccession = filing_row['accessionNumber'].replace('-', '')\nprimary_doc = filing_row['primaryDocument']\nfiling_url = f'https://www.sec.gov/Archives/edgar/data/{int(sample_cik)}/{accession}/{primary_doc}'\nfiling_html = fetch_text(filing_url, headers=SEC_HEADERS)\nfiling_text = html_to_text(filing_html)\nchunks = simple_chunk(filing_text, chunk_size=1800)\nchunk_preview = pd.DataFrame({\n    'ticker': sample_ticker,\n    'cik': sample_cik,\n    'form_type': filing_row['form'],\n    'filing_date': filing_row['filingDate'],\n    'chunk_id': [f'{sample_ticker}-{i:03d}' for i in range(len(chunks[:5]))],\n    'text_preview': chunks[:5],\n})\nchunk_preview.to_parquet(INTERIM_DIR / 'vector_preview' / f'{sample_ticker.lower()}_sec_chunks_preview.parquet', index=False)\nchunk_preview

## 5. News / Narrative + Derived Feature Preview\n\nUse GDELT for a quick free narrative pass, then create a tiny derived feature snapshot so you can sanity-check how the structured and text layers will come together later.

In [ ]:
query = quote(sample_ticker)\ngdelt_url = (\n    'https://api.gdeltproject.org/api/v2/doc/doc'\n    f'?query={query}&mode=artlist&maxrecords=10&format=json'\n)\ngdelt_payload = fetch_json(gdelt_url)\narticles = pd.DataFrame(gdelt_payload.get('articles', []))\narticles.to_parquet(EXTERNAL_DIR / 'gdelt' / f'{sample_ticker.lower()}_news_preview.parquet', index=False)\ndisplay(articles[['title', 'domain', 'seendate', 'url']].head(10) if not articles.empty else pd.DataFrame())\n\nderived_snapshot = pd.DataFrame([{\n    'ticker': sample_ticker,\n    'recent_filings_count': len(recent),\n    'vector_chunk_count_preview': len(chunks),\n    'news_article_count_preview': len(articles),\n    'macro_series_tracked': len(fred_series),\n}])\nderived_snapshot.to_parquet(INTERIM_DIR / 'derived_feature_preview.parquet', index=False)\nderived_snapshot